# B2-023 — Practice p20

*65 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

**Set:** C · **Type:** integrative · **Difficulty:** advanced · **Time budget:** 65 minutes  
**Concepts:** stable-diffusion

## Task

Train a CPU-scale class-conditional latent diffusion model (a frozen autoencoder latent, label-token cross-attention, conditioning dropout) and certify that classifier-free guidance behaves as Session 5 predicts.

**Data and frozen autoencoder.** Load the unit loader with the supplied cell. Define `train_rows()` and `heldout_rows()` with no arguments, returning fresh float32 tensors `generative_data.train_tensor("latent4d")` of shape `(192,4)` and `generative_data.heldout_tensor("latent4d")` of shape `(48,4)`, and `train_labels()` / `heldout_labels()` returning the int64 tensors `generative_data.train_label_tensor("latent4d")` `(192,)` and `generative_data.heldout_label_tensor("latent4d")` `(48,)`, all from the immutable `TRAIN_IDS["latent4d"]` / `HELDOUT_IDS["latent4d"]` in stored order. Held-out rows never enter a training call, a `backward()`, an optimizer step, or a choice of hyperparameter. With `E = generative_data.encoder_matrix()` (shape `(2,4)`) and `LATENT_SCALE = generative_data.LATENT_SCALE` (`0.25`), define the module-level functions `encode(x)` returning `x @ E.T` and `decode(z)` returning `z @ E`. These are frozen: no parameter, no gradient. Clean rows reach the model **only** through `encode`, exactly as returned.

**Schedule.** `make_schedule`, `q_sample` (p09 contract), `p_sample`, and `sample_loop` (p11 contract), with `SCHEDULE = make_schedule(50, 1e-3, 0.2)`.

**Denoiser.** `class CrossAttentionConditioner(nn.Module)` with constructor `CrossAttentionConditioner(query_in, dim, num_classes)` creates, in this order, `self.q_proj = nn.Linear(query_in, dim)`, `self.k_proj = nn.Linear(dim, dim, bias=False)`, `self.v_proj = nn.Linear(dim, dim, bias=False)`, `self.class_tokens = nn.Embedding(num_classes, dim)`, and `self.null_token = nn.Parameter(torch.zeros(dim))`, and stores `self.dim = dim`. `forward(query_feats, labels, keep)` takes float `query_feats` `(B,query_in)`, int64 `labels` `(B,)`, and bool `keep` `(B,)`; it builds `tokens = torch.stack([self.class_tokens(labels), self.null_token.expand(B, dim)], dim=1)` (class token first, shape `(B,2,dim)`), `q = self.q_proj(query_feats).unsqueeze(1)` `(B,1,dim)`, `k = self.k_proj(tokens)`, `v = self.v_proj(tokens)`, `scores = q @ k.transpose(1, 2) / math.sqrt(dim)` `(B,1,2)`, sets the class-token score to `float('-inf')` (with `masked_fill`) for every example whose `keep` is `False`, takes `weights = torch.softmax(scores, dim=-1)`, and returns `(weights @ v).squeeze(1)` of shape `(B,dim)`. `class LatentDenoiser(nn.Module)` creates, in this order, `self.cond = CrossAttentionConditioner(3, 16, 3)` and `self.mlp = nn.Sequential(nn.Linear(19, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))`; `forward(z_t, t_scaled, labels, keep)` computes `c = self.cond(torch.cat([z_t, t_scaled], dim=1), labels, keep)` and returns `self.mlp(torch.cat([z_t, t_scaled, c], dim=1))`. Call `torch.manual_seed(SEED)` immediately before `model = LatentDenoiser()`.

**Training.** `optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`. Write `train_latent_diffusion(model, batch, optimizer)` where `batch = (rows, labels)`. It computes `z0 = encode(rows) * LATENT_SCALE` once, creates `noise_generator = torch.Generator().manual_seed(SEED)` once, and performs **exactly 1500** full-batch updates. Each update draws, in this order, `t = torch.randint(1, 51, (B,), generator=noise_generator)`, `eps = torch.randn(B, 2, generator=noise_generator)`, and `keep = torch.rand(B, generator=noise_generator) >= 0.2` (conditioning dropout $0.2$: a dropped example attends to the null token only); then `optimizer.zero_grad(set_to_none=True)`, `z_t = q_sample(z0, t, eps, SCHEDULE["alpha_bar"])`, `eps_hat = model(z_t, (t.to(torch.float32) / 50).unsqueeze(1), labels, keep)`, `loss = ((eps_hat - eps) ** 2).mean()`, `loss.backward()`, `optimizer.step()`. No shuffling, scheduler, clipping, or early stopping. It returns the list of 1500 Python-float losses. Call it as `trace = train_latent_diffusion(model, (train_rows(), train_labels()), optimizer)`.

**Guided sampling.** Implement `cfg_combine(eps_uncond, eps_cond, w)` with the p12 contract. For a weight `w`, the guided prediction at each reverse step is `cfg_combine(model(z, ts, labels, all_false), model(z, ts, labels, all_true), w)`, where `all_false` is the null-only path used in training. Write `sample_latents(model, labels, w, x_T, noises)` that runs `sample_loop` with this guided prediction under `torch.no_grad()` and returns the final latents. Use `targets = torch.arange(3).repeat_interleave(100)`; `sample_generator = torch.Generator().manual_seed(SEED + 2)`; `x_T = torch.randn(300, 2, generator=sample_generator)`, then `noises = torch.randn(50, 300, 2, generator=sample_generator)`; reuse the **same** `x_T` and `noises` for every `w`. Decode with `x = decode(z / LATENT_SCALE)`.

**Metrics.** With `centers = generative_data.class_centers()` (shape `(3,4)`), `dist(w)` is the mean over the 300 samples of the Euclidean distance from `x` to `centers[targets]`, and `hit(w)` is the fraction whose nearest centre (p08's nearest-centre rule, lowest index on ties) equals the target.

**Certify.**
(A) `len(trace) == 1500`; the mean of the last 50 losses is at most `0.75 *` the mean of the first 50.
(B) **Held-out denoising:** with `eval_generator = torch.Generator().manual_seed(SEED + 1)`, `t_eval = torch.randint(1, 51, (48,), generator=eval_generator)`, then `eps_eval = torch.randn(48, 2, generator=eval_generator)`, the held-out conditional loss (`keep` all `True`, `heldout_labels()`, latents `encode(heldout_rows()) * LATENT_SCALE`) is at most `0.25`, and at most the held-out null-only loss minus `0.03`.
(C) **Continuous guidance criterion:** `dist(0.0) >= dist(1.0) + 1.0` and `dist(1.0) >= dist(1.5) + 0.05`, so the mean distance strictly decreases from $w=0$ to $w=1$ to $w=1.5$.
(D) **Hit rates:** `hit(1.5) >= hit(1.0) > hit(0.0)` and `hit(1.0) >= 0.9`.
(E) **Frozen autoencoder:** `torch.equal(decode(encode(train_rows())), train_rows())`, and `torch.equal(decode((encode(train_rows()) * LATENT_SCALE) / LATENT_SCALE), train_rows())`.
(F) **Split certificate at the clean-row seam.** For the duration of the training call only, wrap the module-level `encode` so that it records every row it receives (restore the original afterwards; held-out evaluation later legitimately uses the same seam). Then assert that no recorded row hashes, with `generative_data.row_sha256`, into the held-out part of `generative_data.ROW_SHA256["latent4d"]` (the hashes at the positions in `generative_data.HELDOUT_IDS["latent4d"]`), and that the recorded hashes include every training row's hash.
These thresholds were set from the frozen-seed reference run with margin; they are not a priori guarantees. The plan's original high guidance weight $w=3$ overshoots in that run (its mean distance rises above $w=1.5$'s), so the certified high weight is pinned to $w=1.5$.

**Interpretation.** Report `dist` and `hit` for $w\in\{0,1,1.5,3\}$ (no assertion for $w=3$), and in four sentences explain the $w=0$ hit rate, why hit rates alone cannot certify the improvement from $w=1$ to $w=1.5$, what happens at $w=3$ and why, and why the null-only path must be trained for (C) to mean anything. The whole notebook must run in under 20 seconds on CPU.

**Required answer-check assertions.** Every item (A)–(F), and the shapes `(192,4)`, `(48,4)`, `(192,)`, `(48,)` of the splits.

In [ ]:
import importlib.util
import math
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)

In [ ]:
# Write the required implementation, probes, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, shape/dtype probe, training certificate, or audit. Use only the permitted literal data and the seeded unit generator, keep train and held-out roles separate, and end coding work with an answer-check section of executable assertions.